In [3]:
import numpy as np
from keras.datasets import mnist
# Load MNIST dataset
(train_X, train_y), (test_X, test_y) = mnist.load_data()

In [4]:
# Convert labels to one-hot encoding
def one_hot_encode(y, num_classes=10):
    one_hot = np.zeros((num_classes, y.shape[0]))
    one_hot[y, np.arange(y.shape[0])] = 1
    return one_hot

In [7]:
# Flatten the input arrays
X_train = train_X.reshape(60000, 784).T  # (784, 60000)
X_test = test_X.reshape(10000, 784).T    # (784, 10000)

Y_train = one_hot_encode(train_y)  # (10, 60000)
Y_test = one_hot_encode(test_y)    # (10, 10000)

In [9]:
# Activation functions
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)  # 1 if Z > 0, else 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))  # Stability trick
    return expZ / np.sum(expZ, axis=0, keepdims=True)

# Cost function
def compute_cost(Y, A2):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(A2 + 1e-8)) / m  # Added epsilon to avoid log(0)
    return np.squeeze(cost)

In [11]:
# Initialize parameters
def init_params():
    # np.random.seed(42) # Fixed Randomness everytime
    W1 = np.random.randn(128, 784) * 0.01  # Small random values
    b1 = np.zeros((128, 1))

    W2 = np.random.randn(64, 128) * 0.01  # Small random values
    b2 = np.zeros((64, 1))
    
    W3 = np.random.randn(10, 64) * 0.01
    b3 = np.zeros((10, 1))
    return W1, b1, W2, b2, W3, b3

In [19]:
# Training function
def compute_grad(X, Y, W1, b1, W2, b2, W3, b3, iters, alpha):
    m = X.shape[1]  # Number of examples

    for _ in range(iters):
        # Forward Pass
        Z1 = np.dot(W1, X) + b1
        A1 = relu(Z1)  # Hidden layer 1 activation

        Z2 = np.dot(W2, A1) + b2
        A2 = relu(Z2)  # Hidden layer 2 activation 

        Z3 = np.dot(W3, A2) + b3
        A3 = relu(Z3)  # Output layer activation (probabilities)

        # Compute Loss Gradient (Softmax + Cross-Entropy)
        dZ3 = A3 - Y  # Gradient of cross-entropy loss
        dW3 = np.dot(dZ3, A2.T) / m
        db3 = np.sum(dZ3, axis=1, keepdims=True) / m

        # Backpropagate through ReLU
        dZ2 = np.dot(W3.T, dZ3) * relu_derivative(Z2)  # Gradient for hidden layer
        dW2 = np.dot(dZ2, A1.T) / m
        db2 = np.sum(dZ2, axis=1, keepdims=True) / m
        
        dZ1 = np.dot(W2.T, dZ2) * relu_derivative(Z1)  # Gradient for hidden layer
        dW1 = np.dot(dZ1, X.T) / m
        db1 = np.sum(dZ1, axis=1, keepdims=True) / m

        # Gradient Descent Updates
        W1 -= alpha * dW1
        b1 -= alpha * db1
        W2 -= alpha * dW2
        b2 -= alpha * db2
        W3 -= alpha * dW3
        b3 -= alpha * db3

        if _ % 10 == 0:
            cost = compute_cost(Y, A3)
            print(f"Iteration {_}: Cost = {cost:.4f}")

    return W1, b1, W2, b2, W3, b3

In [21]:

# Prediction function
def predict(X, W1, b1, W2, b2, W3, b3):
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)  # Hidden layer 1 activation

    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)  # Hidden layer 2 activation 

    Z3 = np.dot(W3, A2) + b3
    A3 = relu(Z3)  # Output layer activation (probabilities)

    predictions = np.argmax(A3, axis=0)  # Get class index with highest probability
    return predictions



In [23]:
# Train the model
W1, b1, W2, b2, W3, b3 = init_params()
W1, b1, W2, b2, W3, b3 = compute_grad(X_train, Y_train, W1, b1, W2, b2, W3, b3, iters=100, alpha=0.01)

Iteration 0: Cost = 6.8566
Iteration 10: Cost = 1.4315
Iteration 20: Cost = 1.1214
Iteration 30: Cost = 1.2674
Iteration 40: Cost = 1.4233
Iteration 50: Cost = 1.0786
Iteration 60: Cost = 0.9152
Iteration 70: Cost = 0.8881
Iteration 80: Cost = 0.8021
Iteration 90: Cost = 0.7650


In [25]:
# Get predictions
predicted_labels = predict(X_test, W1, b1, W2, b2, W3, b3)

In [27]:

# Compute accuracy
accuracy = np.mean(predicted_labels == test_y) * 100
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 90.32%
